In [1]:
import pandas as pd
import numpy as np
from rdkit import Chem
from calc_fps_rm import *

%load_ext autoreload
%autoreload 2

## **Test the model for representation**

In [2]:
import selfies as sf
from rdkit import Chem

smiles2selfie_fn = lambda x: sf.encoder(x)
canonical_fn = lambda smi: Chem.MolToSmiles(Chem.MolFromSmiles(smi), isomericSmiles=False)
smiles_clean_fn = lambda x: ('[' not in x) and ('B' not in x.replace('Br', '')) and ('.' not in x)
len_filter_fn = lambda x: sf.len_selfies(x)<=128

#### **Preprocess drugs**

In [3]:
df = pd.read_csv('/home/rkmvu/Dataset/selfies/drug/mdrug.csv')
_, valid_mask = check_smiles_validity(smiles=df['smiles'].tolist())
df = df[valid_mask].copy()
df['smiles'] = df['smiles'].apply(canonical_fn)
df['selfies'] = df['smiles'].apply(smiles2selfie_fn)

len_mask = df['selfies'].apply(len_filter_fn)
df = df[len_mask]

df_temp = df.copy()
mask = df_temp['smiles'].apply(smiles_clean_fn)
df_temp = df_temp[mask]
df_temp = df_temp.drop_duplicates(subset=['smiles'])
smiles_main = df_temp['smiles'].tolist()
smiles_name = df_temp['name'].tolist()
print(f'Number of smiles: {len(smiles_main)}')
df_temp

<===checking smiles validity===>


100%|██████████| 1381/1381 [00:00<00:00, 4940.11it/s]


Number of smiles: 1089


,name,smiles,InChl,type,selfies
0,Abacavir,Nc1nc(NC2CC2)c2ncn(C3C=CC(CO)C3)c2n1,InChI=1S/C14H18N6O/c15-14-18-12(17-9-2-3-9)11-...,Drug,[N][C][=N][C][Branch1][#Branch1][N][C][C][C][R...
1,Abiraterone,CC(=O)OC1CCC2(C)C(=CCC3C2CCC2(C)C(c4cccnc4)=CC...,InChI=1S/C26H33NO2/c1-17(28)29-20-10-12-25(2)1...,Drug,[C][C][=Branch1][C][=O][O][C][C][C][C][Branch1...
2,Acamprosate,CC(=O)NCCCS(=O)(=O)O,"InChI=1S/C5H11NO4S/c1-5(7)6-3-2-4-11(8,9)10/h2...",Drug,[C][C][=Branch1][C][=O][N][C][C][C][S][=Branch...
3,Acarbose,CC1OC(OC2C(CO)OC(OC3C(CO)OC(O)C(O)C3O)C(O)C2O)...,InChI=1S/C25H43NO18/c1-6-11(26-8-2-7(3-27)12(3...,Drug,[C][C][O][C][Branch2][Ring2][#Branch2][O][C][C...
4,Acebutolol,CCCC(=O)Nc1ccc(OCC(O)CNC(C)C)c(C(C)=O)c1,InChI=1S/C18H28N2O4/c1-5-6-18(23)20-14-7-8-17(...,Drug,[C][C][C][C][=Branch1][C][=O][N][C][=C][C][=C]...
...,...,...,...,...,...
1374,Ziprasidone,O=C1Cc2cc(CCN3CCN(c4nsc5ccccc45)CC3)c(Cl)cc2N1,InChI=1S/C21H21ClN4OS/c22-17-13-18-15(12-20(27...,Drug,[O][=C][C][C][=C][C][Branch2][Ring1][#Branch2]...
1375,Zoledronate,O=P(O)(O)C(O)(Cn1ccnc1)P(=O)(O)O,"InChI=1S/C5H10N2O7P2/c8-5(15(9,10)11,16(12,13)...",Drug,[O][=P][Branch1][C][O][Branch1][C][O][C][Branc...
1377,Zolpidem,Cc1ccc(-c2nc3ccc(C)cn3c2CC(=O)N(C)C)cc1,InChI=1S/C19H21N3O/c1-13-5-8-15(9-6-13)19-16(1...,Drug,[C][C][=C][C][=C][Branch2][Ring1][O][C][N][=C]...
1378,Zonisamide,NS(=O)(=O)Cc1noc2ccccc12,"InChI=1S/C8H8N2O3S/c9-14(11,12)5-7-6-3-1-2-4-8...",Drug,[N][S][=Branch1][C][=O][=Branch1][C][=O][C][C]...


In [4]:
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from rdkit.DataStructs import BulkTanimotoSimilarity

class nneigh_morgan:

    def __init__(self, smiles_list, name_list=None, radius=2, fpSize=2048):
        self.smiles_list = smiles_list
        self.name_list = name_list if name_list is not None else ['None']*len(smiles_list)
        self.name2smiles_df = pd.DataFrame({'smiles':smiles_list, 'name':name_list})
        self.radius = radius
        self.fpSize = fpSize

        self.generator = rdFingerprintGenerator.GetMorganGenerator(radius=self.radius, 
                                                                   fpSize=self.fpSize
                                                                   )
        self.fps = [self.generator.GetFingerprint(Chem.MolFromSmiles(s)) for s in self.smiles_list]

    def get_nearest(self, smiles, n_neigh=10):

        query_fp = self.generator.GetFingerprint(Chem.MolFromSmiles(smiles))
        similarities = BulkTanimotoSimilarity(query_fp, self.fps)
        neighbor_indices = sorted(range(len(similarities)), key=lambda i: similarities[i], reverse=True)
        neighbor_indices = neighbor_indices[1:n_neigh+1]
        nn_smiles = [self.smiles_list[i] for i in neighbor_indices]
        nn_names = [self.smiles2name(_name) for _name in nn_smiles]
        nn_similarities = [similarities[i] for i in neighbor_indices]

        return {'smiles':nn_smiles, 'similarity':nn_similarities, 'names':nn_names}

    def name2smiles(self, name):
        row = self.name2smiles_df[self.name2smiles_df['name'] == name].iloc[0]
        return row['smiles']

    def smiles2name(self, smiles):
        row = self.name2smiles_df[self.name2smiles_df['smiles'] == smiles].iloc[0]
        return row['name']

    def smiles2morgan_fp(self, smiles):
        return calculate_fingerprint_from_smiles(smiles=smiles, fingerprint_name='morgan', n_morgan_bits=1024)

In [5]:
morgan_nn = nneigh_morgan(smiles_list=smiles_main, name_list=smiles_name)

In [6]:
morgan_nn.name2smiles_df

,smiles,name
0,Nc1nc(NC2CC2)c2ncn(C3C=CC(CO)C3)c2n1,Abacavir
1,CC(=O)OC1CCC2(C)C(=CCC3C2CCC2(C)C(c4cccnc4)=CC...,Abiraterone
2,CC(=O)NCCCS(=O)(=O)O,Acamprosate
3,CC1OC(OC2C(CO)OC(OC3C(CO)OC(O)C(O)C3O)C(O)C2O)...,Acarbose
4,CCCC(=O)Nc1ccc(OCC(O)CNC(C)C)c(C(C)=O)c1,Acebutolol
...,...,...
1084,O=C1Cc2cc(CCN3CCN(c4nsc5ccccc45)CC3)c(Cl)cc2N1,Ziprasidone
1085,O=P(O)(O)C(O)(Cn1ccnc1)P(=O)(O)O,Zoledronate
1086,Cc1ccc(-c2nc3ccc(C)cn3c2CC(=O)N(C)C)cc1,Zolpidem
1087,NS(=O)(=O)Cc1noc2ccccc12,Zonisamide


In [7]:
def encode(smiles_list):
    temp = [morgan_nn.smiles2morgan_fp(x) for x in smiles_list]
    return np.stack(temp)

z = encode(smiles_main)
z.shape

(1089, 1024)

In [8]:
df = pd.DataFrame(z, columns=[f'dim_{i}' for i in range(z.shape[1])])
df.insert(0, 'smiles', smiles_main)
df.insert(0, 'selfies', 'None')
df.insert(0, 'name', 'None')
df['name'] = df['smiles'].apply(morgan_nn.smiles2name)
df['selfies'] = df['smiles'].apply(smiles2selfie_fn)

df2 = pd.read_csv('/home/rkmvu/Codes/ICDCIT_submission/results/drug_properties.csv')
df3 = pd.merge(df, df2, on='smiles')
# df3.to_csv('/home/rkmvu/Codes/ICDCIT_submission/results/drug_props_with_embeds_fp.csv', index=False)
df3#.head(2)

,name,selfies,smiles,dim_0,dim_1,dim_2,dim_3,dim_4,dim_5,dim_6,...,VSA_EState3,NHOHCount,NumHDonors,NumHAcceptor,NumRotatableBonds,MolLogP,ATSC1pe,ATSC1are,AATSC1dv,AATSC1are
0,Abacavir,[N][C][=N][C][Branch1][#Branch1][N][C][C][C][R...,Nc1nc(NC2CC2)c2ncn(C3C=CC(CO)C3)c2n1,0,0,0,0,0,0,0,...,12.615719,4,3,7,4,1.09230,-0.478300,-0.657070,1.039823,-0.015645
1,Abiraterone,[C][C][=Branch1][C][=O][O][C][C][C][C][Branch1...,CC(=O)OC1CCC2(C)C(=CCC3C2CCC2(C)C(c4cccnc4)=CC...,0,0,0,0,0,0,0,...,0.000000,0,0,3,2,5.96940,0.296168,0.241524,1.157286,0.003659
2,Acamprosate,[C][C][=Branch1][C][=O][N][C][C][C][S][=Branch...,CC(=O)NCCCS(=O)(=O)O,0,0,0,0,0,0,0,...,2.403611,2,2,3,4,-0.59960,-0.423500,-0.761825,-0.369321,-0.036277
3,Acarbose,[C][C][O][C][Branch2][Ring2][#Branch2][O][C][C...,CC1OC(OC2C(CO)OC(OC3C(CO)OC(O)C(O)C3O)C(O)C2O)...,0,0,0,0,0,0,0,...,135.592960,14,14,19,9,-8.56450,-4.505442,-5.305658,-0.192650,-0.058952
4,Acebutolol,[C][C][C][C][=Branch1][C][=O][N][C][=C][C][=C]...,CCCC(=O)Nc1ccc(OCC(O)CNC(C)C)c(C(C)=O)c1,0,1,0,0,1,0,0,...,15.766019,3,3,5,10,2.36550,-0.276185,-0.373677,1.298077,-0.007186
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1084,Ziprasidone,[O][=C][C][C][=C][C][Branch2][Ring1][#Branch2]...,O=C1Cc2cc(CCN3CCN(c4nsc5ccccc45)CC3)c(Cl)cc2N1,0,0,0,0,0,0,0,...,4.859700,1,1,5,4,3.80900,0.140586,0.027960,0.820617,0.000528
1085,Zoledronate,[O][=P][Branch1][C][O][Branch1][C][O][C][Branc...,O=P(O)(O)C(O)(Cn1ccnc1)P(=O)(O)O,0,0,0,0,0,0,0,...,6.037515,5,5,5,4,-1.11540,-3.912300,-4.806854,-2.436445,-0.184879
1086,Zolpidem,[C][C][=C][C][=C][Branch2][Ring1][O][C][N][=C]...,Cc1ccc(-c2nc3ccc(C)cn3c2CC(=O)N(C)C)cc1,0,0,0,0,0,0,0,...,0.000000,0,0,3,3,3.24884,0.303576,0.245848,1.704815,0.005345
1087,Zonisamide,[N][S][=Branch1][C][=O][=Branch1][C][=O][C][C]...,NS(=O)(=O)Cc1noc2ccccc12,0,0,0,0,0,1,0,...,9.236725,2,1,4,2,0.61630,0.064800,-0.113569,0.317683,-0.004938


In [9]:
smiles_ = morgan_nn.name2smiles('Clozapine')
print(smiles_)
out = morgan_nn.get_nearest(smiles_, n_neigh=2)
out

CN1CCN(C2=Nc3cc(Cl)ccc3Nc3ccccc32)CC1


{'smiles': ['CN1CCN(C2=Nc3ccccc3Oc3ccc(Cl)cc32)CC1',
  'Cc1cc2c(s1)Nc1ccccc1N=C2N1CCN(C)CC1'],
 'similarity': [0.5576923076923077, 0.5370370370370371],
 'names': ['Loxapine', 'Olanzapine']}

## **Nearest Neighbour Analysis**

In [10]:
top_drugs = ["Metformin", "Amoxicillin", "Atorvastatin", "Amlodipine", "Acetaminophen", "Imatinib", "Clozapine", 
             "Ibuprofen", "Azithromycin", "Doxycycline"]
top_drugs = df_temp['name']
n_neigh=50

all_out = {'drug_fp':[], 'nn_idx_fp':[], 'smiles_fp':[], 'names_fp':[]}
for drug in top_drugs:
    _smiles = morgan_nn.name2smiles(drug)
    out = morgan_nn.get_nearest(_smiles, n_neigh=n_neigh)
    out.pop('similarity')
    drug_name = [drug]*n_neigh
    idx = np.arange(1, n_neigh+1)
    out = {'drug':drug_name, 'nn_idx':idx, **out}
    all_out['drug_fp'].extend(out['drug'])
    all_out['nn_idx_fp'].extend(out['nn_idx'])
    all_out['smiles_fp'].extend(out['smiles'])
    all_out['names_fp'].extend(out['names'])

In [ ]:
all_out_df = pd.DataFrame(all_out)
# all_out_df.to_csv('/home/rkmvu/Codes/ICDCIT_submission/results/near_smiles_fp.csv', index=False)
all_out_df


,drug_fp,nn_idx_fp,smiles_fp,names_fp
0,Abacavir,1,COc1nc(N)nc2c1ncn2C1OC(CO)C(O)C1O,Nelarabine
1,Abacavir,2,Nc1nc(Cl)nc2c1ncn2C1CC(O)C(CO)O1,Cladribine
2,Abacavir,3,Nc1ncn(C2CC(O)C(CO)O2)c(=O)n1,Decitabine
3,Abacavir,4,Nc1nc(Cl)nc2c1ncn2C1OC(CO)C(O)C1F,Clofarabine
4,Abacavir,5,Nc1ncnc2c1ncn2C1OC(CO)C(O)C1O,Vidarabine
...,...,...,...,...
54445,Zuclopenthixol,46,CN(C)CCC=C1c2ccccc2COc2ccc(CC(=O)O)cc21,Olopatadine
54446,Zuclopenthixol,47,CN(C)CCCN1c2ccccc2CCc2ccc(Cl)cc21,Clomipramine
54447,Zuclopenthixol,48,O=C1Nc2ccc(Cl)cc2C(c2ccccc2)=NC1O,Oxazepam
54448,Zuclopenthixol,49,O=C1CN=C(c2ccccc2)c2cc(Cl)ccc2N1CC(F)(F)F,Halazepam
